In [ ]:

# ==== SEM & PATH ANALYSIS EN R ====

# 1) Instalar/cargar paquetes necesarios --------------------------------------
pkgs <- c("lavaan", "semPlot", "MASS")
instalar <- setdiff(pkgs, rownames(installed.packages()))
if (length(instalar)) install.packages(instalar, dependencies = TRUE)
lapply(pkgs, library, character.only = TRUE)
options(width = 120)

# -----------------------------------------------------------------------------#
# 2) ANALISIS DE SENDEROS (Path Analysis) con mediación y covariables
#    - Datos simulados reproducibles
#    - Bootstrapping para el efecto indirecto
# -----------------------------------------------------------------------------#

set.seed(123)
N <- 500

# Exógenas correlacionadas (X1, X2)
Sigma <- matrix(c(1, .30,
                  .30, 1), 2, 2)
exog <- MASS::mvrnorm(N, mu = c(0, 0), Sigma = Sigma)
X1 <- exog[, 1]
X2 <- exog[, 2]

# Mediadora y resultado
M <- 0.50*X1 + 0.20*X2 + rnorm(N, 0, 0.70)
Y <- 0.40*X1 + 0.10*X2 + 0.60*M + rnorm(N, 0, 0.70)

datos <- data.frame(X1, X2, M, Y)

# Modelo de senderos con mediación (X1 -> M -> Y) y X2 como covariable
modelo_pa <- '
  # Regresiones
  M ~ a*X1 + g*X2
  Y ~ cprime*X1 + d*X2 + b*M

  # Efectos indirecto y total (mediación)
  indirect := a*b
  total    := cprime + (a*b)
'

# Ajuste con bootstrap para CIs del efecto indirecto
ajuste_pa <- sem(modelo_pa, data = datos, se = "bootstrap", bootstrap = 1000)

cat("\n===== ANALISIS DE SENDEROS (mediación) =====\n")
print(summary(ajuste_pa, fit.measures = TRUE, standardized = TRUE, rsquare = TRUE))
cat("\n-- Estandarizados y CIs bootstrap (perc) --\n")
print(parameterEstimates(ajuste_pa, standardized = TRUE, boot.ci.type = "perc"))
cat("\n-- Medidas de ajuste clave --\n")
print(fitMeasures(ajuste_pa, c("cfi","tli","rmsea","srmr","aic","bic")))

# Diagramas (en pantalla y guardados como PDF)
try({
  semPlot::semPaths(ajuste_pa, what = "std", whatLabels = "std",
                    layout = "tree", nCharNodes = 0, edge.label.cex = .9,
                    sizeMan = 7, sizeLat = 8)
}, silent = TRUE)

try({
  pdf("diagrama_path.pdf", width = 8, height = 6)
  semPlot::semPaths(ajuste_pa, what = "std", whatLabels = "std",
                    layout = "tree", nCharNodes = 0, edge.label.cex = .9,
                    sizeMan = 7, sizeLat = 8)
  dev.off()
  cat("\n[Guardado] diagrama_path.pdf\n")
}, silent = TRUE)


# -----------------------------------------------------------------------------#
# 3) SEM CON VARIABLES LATENTES (lavaan dataset: HolzingerSwineford1939)
#    - CFA de 3 factores + estructura (speed ~ visual + textual)
#    - Estimación con var. latentes estandarizadas y FIML para NA
# -----------------------------------------------------------------------------#

data("HolzingerSwineford1939")  # dataset incluido en lavaan

HS.model <- '
  # Medición (CFA)
  visual  =~ x1 + x2 + x3
  textual =~ x4 + x5 + x6
  speed   =~ x7 + x8 + x9

  # Estructura (ejemplo)
  speed ~ visual + textual

  # (Opcional) covarianzas entre factores
  visual ~~ textual
  visual ~~ speed
  textual ~~ speed
'

ajuste_sem <- sem(HS.model, data = HolzingerSwineford1939,
                  std.lv = TRUE, missing = "fiml")

cat("\n===== SEM con variables latentes =====\n")
print(summary(ajuste_sem, fit.measures = TRUE, standardized = TRUE, rsquare = TRUE))
cat("\n-- Medidas de ajuste clave --\n")
print(fitMeasures(ajuste_sem, c("cfi","tli","rmsea","srmr","aic","bic")))

# (Opcional) índices de modificación para sugerencias de mejora del modelo
cat("\n-- Índices de modificación (>= 10) --\n")
print(head(modindices(ajuste_sem, sort. = TRUE), 20))

# Diagramas (en pantalla y guardados como PDF)
try({
  semPlot::semPaths(ajuste_sem, what = "std", whatLabels = "std",
                    layout = "tree", nCharNodes = 0, edge.label.cex = .9,
                    sizeMan = 7, sizeLat = 8)
}, silent = TRUE)

try({
  pdf("diagrama_sem.pdf", width = 8, height = 6)
  semPlot::semPaths(ajuste_sem, what = "std", whatLabels = "std",
                    layout = "tree", nCharNodes = 0, edge.label.cex = .9,
                    sizeMan = 7, sizeLat = 8)
  dev.off()
  cat("\n[Guardado] diagrama_sem.pdf\n")
}, silent = TRUE)

cat("\n*** Listo: tienes resultados, métricas de ajuste, efectos (directo/indirecto/total), R^2 y diagramas.\n")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘GPArotation’, ‘gridExtra’, ‘htmlTable’, ‘viridis’, ‘Formula’, ‘arm’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘openxlsx’, ‘RUnit’, ‘mvtnorm’, ‘proto’, ‘psych’, ‘Hmisc’, ‘jpeg’, ‘png’, ‘reshape2’, ‘glasso’, ‘fdrtool’, ‘gtools’, ‘pbapply’, ‘abind’, ‘mi’, ‘lme4’, ‘carData’, ‘kutils’, ‘RcppParallel’, ‘RcppEigen’, ‘StanHeaders’, ‘BH’, ‘rpf’, ‘gsubfn’, ‘coda’, ‘texreg’, ‘pander’, ‘fastDummies’, ‘checkmate’, ‘mnormt’, ‘pbivnorm’, ‘numDeriv’, ‘quadprog’, ‘qgraph’, ‘sem’, ‘plyr’, ‘XML’, ‘igraph’, ‘lisrelToR’, ‘rockchalk’, ‘colorspace’, ‘corpcor’, ‘OpenMx’, ‘MplusAutomation’


Warning message in install.packages(instalar, dependencies = TRUE):
“installation of package ‘RcppEigen’ had non-zero exit status”
Warning message in install.packages(instalar, dependencies = TRUE):
“installation of package ‘StanHeaders’ had non-zero exit status”
Warning message in install.pa

ERROR: Error in FUN(X[[i]], ...): there is no package called ‘semPlot’
